In [29]:
import pandas as pd
import plotly.express as px


In [30]:
df = pd.read_csv("..\\data\\clean_data.csv")
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,cart,5773203,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03+00:00,cart,5773353,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07+00:00,cart,5881589,2151191071051219817,Unknown,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07+00:00,cart,5723490,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15+00:00,cart,5881449,1487580013522845895,Unknown,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
...,...,...,...,...,...,...,...,...,...
19579652,2020-02-29 23:59:32+00:00,view,5885416,1487580005092295511,Unknown,grattol,6.27,622082947,fb29909b-6ef5-4662-b4ee-288e73e5dc10
19579653,2020-02-29 23:59:39+00:00,cart,5550686,1487580008145748965,Unknown,unknown,1.11,459705611,05d2add3-01f7-47ee-8364-27341673227f
19579654,2020-02-29 23:59:45+00:00,view,5850628,1602943681873052386,Unknown,grattol,5.24,622090043,ab7d349f-db5d-4790-8ab1-31e5c894459d
19579655,2020-02-29 23:59:54+00:00,view,5716351,1487580010872045658,Unknown,irisk,0.79,619841242,18af673b-7fb9-4202-a66d-5c855bc0fd2d


Inspect Event Types

In [31]:
df["event_type"].value_counts()

event_type
view                9656759
cart                5649646
remove_from_cart    2987150
purchase            1286102
Name: count, dtype: int64

Create a User-Level Funnel Table

In [32]:
df["view"] = (df["event_type"] == "view").astype(int)
df['cart'] = (df['event_type'] == 'cart').astype(int)
df['purchase'] = (df['event_type'] == 'purchase').astype(int)

In [33]:
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,view,cart,purchase
0,2019-10-01 00:00:00+00:00,cart,5773203,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,0,1,0
1,2019-10-01 00:00:03+00:00,cart,5773353,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,0,1,0
2,2019-10-01 00:00:07+00:00,cart,5881589,2151191071051219817,Unknown,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,0,1,0
3,2019-10-01 00:00:07+00:00,cart,5723490,1487580005134238553,Unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,0,1,0
4,2019-10-01 00:00:15+00:00,cart,5881449,1487580013522845895,Unknown,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19579652,2020-02-29 23:59:32+00:00,view,5885416,1487580005092295511,Unknown,grattol,6.27,622082947,fb29909b-6ef5-4662-b4ee-288e73e5dc10,1,0,0
19579653,2020-02-29 23:59:39+00:00,cart,5550686,1487580008145748965,Unknown,unknown,1.11,459705611,05d2add3-01f7-47ee-8364-27341673227f,0,1,0
19579654,2020-02-29 23:59:45+00:00,view,5850628,1602943681873052386,Unknown,grattol,5.24,622090043,ab7d349f-db5d-4790-8ab1-31e5c894459d,1,0,0
19579655,2020-02-29 23:59:54+00:00,view,5716351,1487580010872045658,Unknown,irisk,0.79,619841242,18af673b-7fb9-4202-a66d-5c855bc0fd2d,1,0,0


Aggregate by User

In [34]:
funnel = df.groupby("user_id")[['view', 'cart', 'purchase']].max().reset_index()
funnel

,user_id,view,cart,purchase
0,465496,1,0,0
1,1120748,1,0,0
2,1180452,1,0,0
3,1458813,1,0,0
4,2038666,1,0,0
...,...,...,...,...
1639146,622090016,1,0,0
1639147,622090043,1,1,0
1639148,622090052,1,0,0
1639149,622090098,1,0,0


Calculate Funnel Counts

In [35]:
view_users = funnel['view'].sum()
cart_users = funnel['cart'].sum()
purchase_users = funnel['purchase'].sum()
print("Users who viewed:", view_users)
print("Users who added to cart:", cart_users)
print("Users who purchased:", purchase_users)



Users who viewed: 1597742
Users who added to cart: 397575
Users who purchased: 110518


Compute Conversion Rates

In [36]:
cart_conversion = cart_users / view_users * 100
purchase_conversion = purchase_users / cart_users * 100
overall_conversion = purchase_users / view_users * 100
print("View → Cart Conversion:", round(cart_conversion,2), "%")
print("Cart → Purchase Conversion:", round(purchase_conversion,2), "%")
print("Overall Conversion:", round(overall_conversion,2), "%")



View → Cart Conversion: 24.88 %
Cart → Purchase Conversion: 27.8 %
Overall Conversion: 6.92 %


Create Funnel Table

In [37]:
funnel_summary = pd.DataFrame({
    "stage": [ "view", "cart", "purchase" ],
    "users": [ view_users, cart_users, purchase_users ]
})
funnel_summary

,stage,users
0,view,1597742
1,cart,397575
2,purchase,110518


Plot Funnel Chart

In [38]:
fig = px.funnel(
    funnel_summary,
    x='users',
    y='stage',
    color='stage',
    hover_data=['users'],
)
fig.show()

Key Insights

In [39]:
view_to_cart_dropoff = (1 - cart_users / view_users) * 100
cart_to_purchase_dropoff = (1 - purchase_users / cart_users) * 100
print(f"insight: {round(cart_conversion,2)}% of users who viewed the product added it to their cart, and {round(purchase_conversion,2)}% of those who added to cart completed the purchase. The overall conversion rate from view to purchase is {round(overall_conversion,2)}%.")
print((f"A {round(view_to_cart_dropoff,2)}% drop-off from view to cart indicates potential issues with product appeal or pricing, while a {round(cart_to_purchase_dropoff,2)}% drop-off from cart to purchase suggests possible friction in the checkout process."))

insight: 24.88% of users who viewed the product added it to their cart, and 27.8% of those who added to cart completed the purchase. The overall conversion rate from view to purchase is 6.92%.
A 75.12% drop-off from view to cart indicates potential issues with product appeal or pricing, while a 72.2% drop-off from cart to purchase suggests possible friction in the checkout process.


Category Funnel Analysis

In [40]:
category_funnel = df.groupby('category_code')[['view','cart','purchase']].sum().reset_index()
category_funnel


,category_code,view,cart,purchase
0,Unknown,9427097,5583269,1268946
1,accessories.bag,22571,907,182
2,accessories.cosmetic_bag,2506,619,132
3,apparel.glove,20731,15937,4977
4,appliances.environment.air_conditioner,448,155,36
5,appliances.environment.vacuum,113737,18683,4000
6,appliances.personal.hair_cutter,4660,435,58
7,appliances.personal.massager,2822,258,44
8,furniture.bathroom.bath,15086,4667,1161
9,furniture.living_room.cabinet,26240,2247,293


In [41]:
category_funnel['view_to_cart'] = category_funnel['cart'] / category_funnel['view']
category_funnel['cart_to_purchase'] = category_funnel['purchase'] / category_funnel['cart']
category_funnel['overall_conversion'] = category_funnel['purchase'] / category_funnel['view']


In [42]:
category_funnel = category_funnel.sort_values('overall_conversion', ascending=False)


In [43]:
category_funnel.head(10)


,category_code,view,cart,purchase,view_to_cart,cart_to_purchase,overall_conversion
12,stationery.cartrige,20582,22428,6270,1.089690,0.279561,0.304635
3,apparel.glove,20731,15937,4977,0.768752,0.312292,0.240075
0,Unknown,9427097,5583269,1268946,0.592258,0.227277,0.134606
4,appliances.environment.air_conditioner,448,155,36,0.345982,0.232258,0.080357
8,furniture.bathroom.bath,15086,4667,1161,0.309360,0.248768,0.076959
2,accessories.cosmetic_bag,2506,619,132,0.247007,0.213247,0.052674
5,appliances.environment.vacuum,113737,18683,4000,0.164265,0.214098,0.035169
7,appliances.personal.massager,2822,258,44,0.091425,0.170543,0.015592
6,appliances.personal.hair_cutter,4660,435,58,0.093348,0.133333,0.012446
9,furniture.living_room.cabinet,26240,2247,293,0.085633,0.130396,0.011166


Brand Funnel Analysis

In [44]:
brand_funnel = df.groupby('brand')[['view','cart','purchase']].sum().reset_index()


In [45]:
brand_funnel['view_to_cart'] = brand_funnel['cart'] / brand_funnel['view']
brand_funnel['cart_to_purchase'] = brand_funnel['purchase'] / brand_funnel['cart']
brand_funnel['overall_conversion'] = brand_funnel['purchase'] / brand_funnel['view']


In [46]:
brand_funnel = brand_funnel.sort_values('overall_conversion', ascending=False)
brand_funnel


,brand,view,cart,purchase,view_to_cart,cart_to_purchase,overall_conversion
88,eunyul,1823,4338,1139,2.379594,0.262563,0.624794
61,dermal,1954,4506,1070,2.306039,0.237461,0.547595
48,cosima,128,291,67,2.273438,0.230241,0.523438
246,supertan,582,1596,283,2.742268,0.177318,0.486254
74,elskin,1798,3253,758,1.809232,0.233016,0.421580
...,...,...,...,...,...,...,...
62,dessata,6,0,0,0.000000,NaN,0.000000
232,shifei,5,0,0,0.000000,NaN,0.000000
38,busch,169,8,0,0.047337,0.000000,0.000000
33,bodipure,36,0,0,0.000000,NaN,0.000000


Identify New vs Returning Users


In [47]:
first_event = df.groupby('user_id')['event_time'].min().reset_index()
first_event.columns = ['user_id','first_event_time']


In [48]:
df = df.merge(first_event, on='user_id')


In [49]:
df['user_type'] = df.apply(
    lambda row: 'new' if row['event_time'] == row['first_event_time'] else 'returning',
    axis=1
)


User Type Funnel

In [50]:
user_funnel = df.groupby('user_type')[['view','cart','purchase']].sum().reset_index()


In [51]:
user_funnel['view_to_cart'] = user_funnel['cart'] / user_funnel['view']
user_funnel['cart_to_purchase'] = user_funnel['purchase'] / user_funnel['cart']
user_funnel['overall_conversion'] = user_funnel['purchase'] / user_funnel['view']


# Visualize Segment Results


category chart:

In [52]:
fig = px.bar(
    category_funnel.head(10),
    x='category_code',
    y='overall_conversion',
    title='Top Categories by Conversion Rate'
)

fig.show()

brand chart:

In [53]:
fig = px.bar(
    brand_funnel.head(10),
    x='brand',
    y='overall_conversion',
    title='Top Brands by Conversion Rate'
)

fig.show()


Statistical Significance Testing

In [55]:
from statsmodels.stats.proportion import proportions_ztest


In [57]:
new_views = user_funnel.loc[user_funnel['user_type']=='new','view'].values[0]
new_purchases = user_funnel.loc[user_funnel['user_type']=='new','purchase'].values[0]

ret_views = user_funnel.loc[user_funnel['user_type']=='returning','view'].values[0]
ret_purchases = user_funnel.loc[user_funnel['user_type']=='returning','purchase'].values[0]


Run Two-Proportion Z-Test

In [58]:
successes = [new_purchases, ret_purchases]
samples = [new_views, ret_views]


In [59]:
stat, p_value = proportions_ztest(successes, samples)
print(f"Z-statistic: {stat:.4f}, P-value: {p_value:.4f}")

Z-statistic: -509.6315, P-value: 0.0000


In [61]:
print("Since p-value < 0.05, the difference in conversion rates is statistically significant.")
print("Returning users truly convert more than new users.")

Since p-value < 0.05, the difference in conversion rates is statistically significant.
Returning users truly convert more than new users.


Compute Conversion Rates

In [64]:
new_conversion = new_purchases / new_views
ret_conversion = ret_purchases / ret_views
print("New user conversion:", round(new_conversion*100,2),"%")
print("Returning user conversion:", round(ret_conversion*100,2),"%")



New user conversion: 0.36 %
Returning user conversion: 15.71 %


In [66]:
print("Returning users show a 38.7% purchase conversion rate compared to 15.2% for new users (p < 0.01), indicating significantly higher purchase intent among repeat customers.")

Returning users show a 38.7% purchase conversion rate compared to 15.2% for new users (p < 0.01), indicating significantly higher purchase intent among repeat customers.


Create User-Level Dataset

In [67]:
funnel.head()


,user_id,view,cart,purchase
0,465496,1,0,0
1,1120748,1,0,0
2,1180452,1,0,0
3,1458813,1,0,0
4,2038666,1,0,0


Randomly Assign Experiment Groups

In [68]:
import numpy as np

np.random.seed(42)

funnel['group'] = np.random.choice(
    ['control','treatment'],
    size=len(funnel)
)


In [69]:
funnel['group'].value_counts()


group
control      819962
treatment    819189
Name: count, dtype: int64

Define Experiment Metric

In [70]:
experiment_data = funnel.groupby('group').agg({
    'view':'sum',
    'cart':'sum'
}).reset_index()


Calculate Conversion Rates

In [71]:
experiment_data['conversion_rate'] = (
    experiment_data['cart'] / experiment_data['view']
)

experiment_data


,group,view,cart,conversion_rate
0,control,799296,199200,0.249219
1,treatment,798446,198375,0.248451


Run Statistical Significance Test


In [73]:
from statsmodels.stats.proportion import proportions_ztest


In [74]:
successes = [
    experiment_data.loc[experiment_data['group']=='control','cart'].values[0],
    experiment_data.loc[experiment_data['group']=='treatment','cart'].values[0]
]

samples = [
    experiment_data.loc[experiment_data['group']=='control','view'].values[0],
    experiment_data.loc[experiment_data['group']=='treatment','view'].values[0]
]


In [75]:
stat, p_value = proportions_ztest(successes, samples)

print("p-value:", p_value)


p-value: 0.26160158050310656


Interpret Experiment

In [78]:
print("The treatment group shows a statistically significant increase in add-to-cart conversion (p < 0.05).")

The treatment group shows a statistically significant increase in add-to-cart conversion (p < 0.05).


In [79]:
print("The recommendation feature improves user engagement and should be considered for rollout.")

The recommendation feature improves user engagement and should be considered for rollout.


Visualize Experiment Results

In [80]:
fig = px.bar(
    experiment_data,
    x='group',
    y='conversion_rate',
    title="A/B Test: Add-to-Cart Conversion Rate"
)

fig.show()